## Part II: Practice the NLP model to classify data stories 

#### Step-1 

Preprocess the Text

We will:
- Convert to lowercase.
- Remove punctuation.
- Tokenize text.
- Remove stop words.
- Apply lemmatization.

In [3]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

# Step 1: Load Dataset
# Replace 'data_file.csv' with your actual dataset file
data = pd.read_csv('data_stories_one_shot.csv')

# Display first few rows to understand structure
print(data.head())

# Step 2: Define Preprocessing Function
def preprocess_text(text):
    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))
    text = text.lower()  # Convert to lowercase
    text = re.sub(r'[^\w\s]', '', text)  # Remove punctuation
    tokens = word_tokenize(text)  # Tokenize text
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]  # Remove stopwords & lemmatize
    return ' '.join(tokens)

# Step 3: Apply Preprocessing
# Replace 'text_column' with the actual name of the text column
data['processed_text'] = data['Sentence'].apply(preprocess_text)

# Step 4: Verify Preprocessing
print(data[['Sentence', 'processed_text']].head())


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Unknown1\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Unknown1\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Unknown1\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


  Plot_Name  Stage  Quality                                           Sentence
0  walk dog      1      1.0              This is a line chart with error bars.
1  walk dog      1      1.0                     The chart title is 'Walk dog'.
2  walk dog      1      1.0              The y-axis represents 'Mean anxiety'.
3  walk dog      1      1.0  The x-axis indicates conditions such as 'Basel...
4  walk dog      1      1.0  The chart compares mean anxiety levels with an...
                                            Sentence  \
0              This is a line chart with error bars.   
1                     The chart title is 'Walk dog'.   
2              The y-axis represents 'Mean anxiety'.   
3  The x-axis indicates conditions such as 'Basel...   
4  The chart compares mean anxiety levels with an...   

                                      processed_text  
0                               line chart error bar  
1                               chart title walk dog  
2                      y

#### Step 2: Feature Extraction using TF-IDF.
- The goal is to convert the cleaned text into numerical features using TF-IDF (Term Frequency-Inverse Document Frequency).

In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Step 1: Initialize TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=1000)  # Set a limit on features for simplicity

# Step 2: Apply TF-IDF on the processed text
X = tfidf_vectorizer.fit_transform(data['processed_text'])  # Features
y = data['Stage']  # Replace 'label' with the actual column name for target labels

# Step 3: Check the shape of the resulting feature matrix
print("TF-IDF matrix shape:", X.shape)
print("Target labels shape:", y.shape)


TF-IDF matrix shape: (130, 408)
Target labels shape: (130,)


#### Step 3: Model Training
The goal is to train the classification models (Logistic Regression, SVM, Naive Bayes). and validate using cross validation and lean one plot out.

In [25]:
from sklearn.model_selection import train_test_split, cross_val_score, LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Map Stage values to binary (1 for "Show" and 0 for "Tell" [Stage 2, 3])
data['binary_stage'] = data['Stage'].apply(lambda x: 1 if x == 1 else 0)

# Step 1: Split the data
X = data['processed_text']
y = data['binary_stage']

# Vectorize the text data using TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=1000)
X_tfidf = tfidf_vectorizer.fit_transform(data['processed_text'])

# Train-Test split
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(kernel='linear', probability=True),
    "Naive Bayes": MultinomialNB()
}

# Adding cross-validation and leave-one-out validation scores to the consolidated table
all_metrics_with_cv_loo = []

for name, model in models.items():
    # Fit the model and make predictions
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    # Calculate standard metrics
    auc = roc_auc_score(y_test, y_pred_proba)
    accuracy = accuracy_score(y_test, predictions)
    precision = precision_score(y_test, predictions)
    recall = recall_score(y_test, predictions)
    f1 = f1_score(y_test, predictions)
    
    # Cross-validation scores
    cv_scores = cross_val_score(model, X_tfidf, y, cv=5)
    cv_mean_accuracy = np.mean(cv_scores)
    cv_std_dev = np.std(cv_scores)
    
    # Leave-One-Out Validation
    loo = LeaveOneOut()
    loo_accuracies = []
    for train_index, test_index in loo.split(X_tfidf):
        X_train_loo, X_test_loo = X_tfidf[train_index], X_tfidf[test_index]
        y_train_loo, y_test_loo = y.iloc[train_index], y.iloc[test_index]
        model.fit(X_train_loo, y_train_loo)
        loo_accuracies.append(model.score(X_test_loo, y_test_loo))
    loo_mean_accuracy = np.mean(loo_accuracies)
    
    # Append metrics to the list
    all_metrics_with_cv_loo.append({
        "Model": name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "AUC": auc,
        "CV Mean Accuracy": cv_mean_accuracy,
        "CV Std Dev": cv_std_dev,
        "LOO Mean Accuracy": loo_mean_accuracy
    })

# Convert to DataFrame
metrics_with_cv_loo_df = pd.DataFrame(all_metrics_with_cv_loo)

# Display the results
# tools.display_dataframe_to_user("Metrics with Cross-Validation and Leave-One-Out Scores", metrics_with_cv_loo_df)
print(metrics_with_cv_loo_df)


                 Model  Accuracy  Precision    Recall  F1 Score       AUC  \
0  Logistic Regression  0.846154   0.842105  0.941176  0.888889  0.921569   
1                  SVM  0.807692   0.833333  0.882353  0.857143  0.928105   
2          Naive Bayes  0.923077   0.894737  1.000000  0.944444  0.960784   

   CV Mean Accuracy  CV Std Dev  LOO Mean Accuracy  
0          0.623077    0.028782           0.723077  
1          0.807692    0.054393           0.800000  
2          0.746154    0.079197           0.769231  


The comparison of Logistic Regression, SVM, and Naive Bayes models reveals notable distinctions in performance metrics. Naive Bayes demonstrated the highest overall accuracy (92.31%) and recall (100%), indicating its strong ability to correctly classify positive instances. However, Logistic Regression and SVM achieved competitive precision and F1 scores, with Logistic Regression excelling in cross-validation stability (lowest standard deviation) but showing relatively lower cross-validation accuracy compared to SVM. Leave-One-Out (LOO) validation highlights that SVM had the highest mean accuracy (80%), showcasing its robust performance across individual samples. Overall, Naive Bayes emerged as the most accurate model, but SVM maintained balanced and consistent performance across various metrics.